<a href="https://colab.research.google.com/github/chaihajare/rag_document_assistant/blob/main/rag_document_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the Groq SDK
#!pip install -q groq

print('done')

done


In [2]:
from google.colab import userdata
from groq import Groq

# Load your API key securely from Colab Secrets
api_key = userdata.get('GROQ_API_KEY').strip()
client = Groq(api_key=api_key)

# Quick test call
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working!"}
    ]
)

print(response.choices[0].message.content)

Hello. I'm working and ready to assist you with any questions or tasks you may have. How can I help you today?


In [3]:
#!pip install -q feedparser
print('done')

done


In [4]:
import feedparser

feeds = {
    "BBC Football": "http://feeds.bbci.co.uk/sport/football/rss.xml",
    "ESPN Soccer": "https://www.espn.com/espn/rss/soccer/news",
    "Guardian Football": "https://www.theguardian.com/football/rss"
}

all_articles = []

for source, feed_url in feeds.items():
    feed = feedparser.parse(feed_url)
    print(f"{source}: {len(feed.entries)} articles found")
    for entry in feed.entries[:15]:
        all_articles.append({
            "source": source,
            "title": entry.title,
            "summary": entry.get("summary", ""),
            "link": entry.link,
            "published": entry.get("published", "")
        })

# Filter to only World Cup-related articles
worldcup_keywords = ["world cup", "fifa", "round of", "knockout", "group stage"]
worldcup_articles = [
    a for a in all_articles
    if any(kw in (a['title'] + a['summary']).lower() for kw in worldcup_keywords)
]

print(f"\nTotal articles: {len(all_articles)}")
print(f"World Cup-related: {len(worldcup_articles)}")
print(f"\nSample: {worldcup_articles[0] if worldcup_articles else 'None found - may need broader feeds'}")

BBC Football: 83 articles found
ESPN Soccer: 0 articles found
Guardian Football: 66 articles found

Total articles: 30
World Cup-related: 19

Sample: {'source': 'BBC Football', 'title': "France superstars thriving thanks to Deschamps' bold changes", 'summary': "Why Didier Deschamps' changes to personnel and formation could propel 2022 runners-up France to go one better in this World Cup.", 'link': 'https://www.bbc.co.uk/sport/football/articles/cg53ggg03nno?at_medium=RSS&at_campaign=rss', 'published': 'Tue, 30 Jun 2026 11:18:24 GMT'}


In [5]:
# Combine title + summary into one text block per article, with metadata
documents = []

for article in worldcup_articles:
    text = f"{article['title']}. {article['summary']}"
    documents.append({
        "text": text,
        "source": article['source'],
        "title": article['title'],
        "link": article['link'],
        "published": article['published']
    })

print(f"Prepared {len(documents)} documents for embedding")
print(f"\nExample:\n{documents[0]['text']}")

Prepared 19 documents for embedding

Example:
France superstars thriving thanks to Deschamps' bold changes. Why Didier Deschamps' changes to personnel and formation could propel 2022 runners-up France to go one better in this World Cup.


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [doc['text'] for doc in documents]
print("Generating embeddings...")
embeddings = embed_model.encode(texts, show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings shape: (19, 384)


In [7]:
#!pip install -q faiss-cpu
#print('done')

import faiss
import numpy as np

# Convert embeddings to the format FAISS expects
embeddings_np = np.array(embeddings).astype('float32')

# Build a simple FAISS index (L2 distance)
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

print(f"FAISS index built with {index.ntotal} articles")

FAISS index built with 19 articles


In [8]:
def retrieve_relevant_articles(query, top_k=5):
    query_embedding = embed_model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(documents[idx])
    return results

test_results = retrieve_relevant_articles("Tottenham transfer news")
for r in test_results:
    print(f"[{r['source']}] {r['title']}")

[BBC Football] Germany fans in need of hope as prospect of Klopp looms
[Guardian Football] Côte d’Ivoire v Norway: World Cup 2026 last 32 – live
[Guardian Football] Every World Cup needs a cult hero: 2026 has given us touchline dreamboat Sebastián Beccacece
[Guardian Football] Celebrations and bottle-throwing on Dutch streets after dramatic Morocco win
[Guardian Football] David Squires on … World Cup penalty pain for Germany and the Netherlands


In [9]:
def ask_match_day_buzz(question, persona="excited match commentator", top_k=5):
    relevant_docs = retrieve_relevant_articles(question, top_k=top_k)

    context = "\n\n".join([
        f"[{doc['source']}] {doc['text']} (Published: {doc['published']})"
        for doc in relevant_docs
    ])

    system_prompt = f"""You are a football news assistant with the personality of a {persona}.
Answer the user's question using ONLY the information in the provided articles below.
Stay fully in character as a {persona} - be entertaining, but don't make up facts not in the articles.
If the articles don't contain enough info to answer, say so in character.
Always mention which source(s) your info came from.

ARTICLES:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content

answer = ask_match_day_buzz("What happened in today's Round of 32 matches?")
print(answer)

FOLKS, WE'VE GOT A PROBLEM HERE! I've been scanning the articles, and I've got to tell you, there's NO MENTION of Tottenham's transfer business! It's like they're not even on the radar! I've checked all the sources, from the Guardian to the BBC, and nothing's popping up. It's a blank slate, folks!

Now, I know what you're thinking, "What about all the juicy transfer rumors?" Well, I've got nothing for you, mate! It's like the transfer window is a closely guarded secret, and nobody's spilling the beans!

So, if you're looking for the latest on Tottenham's transfer business, I'm afraid you'll have to keep looking. But don't worry, I'll be here, keeping an eye on things, and as soon as something breaks, I'll be shouting it from the rooftops!

Source: NONE OF THE PROVIDED ARTICLES MENTION TOTTENHAM'S TRANSFER BUSINESS!


In [11]:

answer = ask_match_day_buzz("What happened in Germany's team last match")
print(answer)

FOLKS, WE'VE GOT A STUNNING UPSET HERE! According to the BBC Football article "Germany beaten on penalties as Paraguay reach last 16", Germany's last match was an absolute thriller! They played Paraguay in the World Cup, and after a 1-1 draw after extra time, the match went to PENALTIES!

And, oh, what a shocker! Paraguay came out on top, winning the penalty shootout 4-3! That's right, folks, Germany, the four-time World Cup winners, are OUT of the tournament! I'm still trying to process it, but the stats are clear: Paraguay's victory marks the end of Germany's 50-year penalty shootout streak, as reported by the Guardian Football article "Panenka to Paraguay: Germany’s 50-year penalty shootout streak is over".

What a match, folks! I'm still reeling from the excitement! This information comes from the BBC Football article "Germany beaten on penalties as Paraguay reach last 16" and the Guardian Football article "Panenka to Paraguay: Germany’s 50-year penalty shootout streak is over".
